# Example 5 - Sea-Level Fingerprints from GRACE (SlrGRACE)

## What is ISSM-SESAW?

ISSM-SESAW ("Sea-level Equation / Solid-Earth And Water") is the line of
work in pyISSM that couples ice/water mass redistribution to its effect on
**sea level itself** — as opposed to the rest of pyISSM, which is mostly
concerned with computing *how much and where* ice mass changes (ice flow,
calving, surface mass balance). This notebook is the first full worked
example of that machinery: it takes an observed pattern of mass change
(from the GRACE satellite gravity mission) and computes the resulting
"sea-level fingerprint" — the spatially-varying sea-level response, which is
**not** the same everywhere on the globe.

### Why isn't sea-level rise uniform?

A naive model of sea-level rise — "melt some ice, spread the water evenly
over the ocean" — is wrong, and wrong in a large, measurable way. When mass
moves from one place to another (e.g. ice melts in Greenland and the
meltwater is redistributed across the ocean), three physical effects combine
to produce a highly non-uniform sea-level response, together known as the
**GRD problem** (Gravitational, Rotational, and solid-Earth Deformation):

1. **Solid-Earth deformation** — the crust itself sinks under a growing load
   and rebounds under a shrinking one (the same physics as post-glacial
   rebound), changing the seafloor's shape beneath the water.
2. **Gravitational self-attraction** — ice sheets are massive enough to
   gravitationally pull the nearby ocean towards them. When an ice sheet
   loses mass, its gravitational pull weakens, and *sea level actually falls*
   in the surrounding region (typically within ~2000 km) even as it rises
   elsewhere — one of the more counter-intuitive results in sea-level
   science.
3. **Rotational feedback** — redistributing mass across the planet shifts
   its rotation (moment of inertia) axis slightly, which in turn perturbs
   the centrifugal contribution to sea level globally.

The combined spatial pattern from all three effects, per unit of mass lost
from a given source, is that source's "fingerprint" — hence the name. This
notebook solves the sea-level equation that produces that fingerprint for
real GRACE-observed mass changes.

### What this notebook does, step by step

1. **Mesh** — build a global spherical mesh and refine it near the
   coastline, where the sea-level response varies most sharply.
2. **Forcing** — load a single monthly GRACE mass-change epoch and convert
   it into an ice-equivalent surface load.
3. **Parameterization** — assign Love numbers (the material response
   functions of the solid Earth), solid-earth solver settings, and
   timestepping, then verify the model is fully consistent and ready to
   solve. **This is the main locally-verifiable milestone** — everything up
   to here can be checked without a cluster.
4. **Solve** (single epoch) — dispatch to a compiled ISSM binary on a
   cluster (NCI Gadi). Requires the user's own cluster credentials.
5. **Plot** the single-epoch sea-level and water-height change.
6. **Solve** a multi-month transient series (repeats Steps 2–4 across many
   GRACE epochs).
7. **Plot** the transient results and the global-mean sea-level (GMSL) time
   series.

### Provenance

This is a from-scratch pyISSM reimplementation of ISSM's MATLAB `SlrGRACE`
tutorial (https://issm.jpl.nasa.gov/documentation/tutorials/sealevelfingerprints/).
The upstream MATLAB/classic-ISSM code was used only as a reference for *what*
each step needs to compute, never *how* — see the pyISSM coding standards for
why a from-scratch rewrite is preferred over a line-by-line port throughout
this project.

In [ ]:
import numpy as np
import pyissm

---
## Setup your modelling environment

This tutorial needs a local copy of the GRACE Tellus land mass-concentration
product (JPL RL05.1, `DSTvSCS1411`). It is a satellite gravimetry data
product, not something pyISSM ships or generates — download it yourself and
point `DATA_DIR` at the folder containing it.

In [ ]:
# GRACE Tellus land mass-concentration product (JPL RL05.1 DSTvSCS1411) -
# the satellite gravimetry data this tutorial's mass-change forcing comes
# from. Not distributed with pyISSM: update this path to your own copy.
DATA_DIR = '/path/to/GRACE_and_supporting_datasets'
GRACE_NC = f'{DATA_DIR}/GRCTellus.JPL.200204_201701.LND.RL05_1.DSTvSCS1411.nc'

---
## 1. Model mesh: global sphere + coastal refinement

A sea-level fingerprint is a *global* quantity — mass lost in one place
changes sea level everywhere, not just locally — so unlike most other pyISSM
tutorials (which mesh a single glacier or ice shelf), this model's domain is
the entire planet. We build that domain in three steps:

1. Generate an initial, coarse, uniform-resolution global mesh.
2. Classify every vertex of that mesh as ocean or land.
3. Rebuild the mesh a second time, now targeting a *variable* resolution
   that is finest near the coastline (where the sea-level response, ocean
   loading, and the land/ocean boundary itself all vary sharply over short
   distances) and coarsest over the open ocean and deep continental
   interiors (where a much coarser mesh captures the same physics for far
   less computational cost).

In [ ]:
# Radius of a sphere with the same volume as the (slightly oblate) real
# Earth, in km - matches the constant used throughout the upstream tutorial.
RADIUS_KM = 6371.012

# Every pyISSM model starts from an empty Model object; mesh, geometry,
# and every other sub-class are populated onto it step by step.
md = pyissm.model.Model()

# Build the initial global mesh: a uniform-resolution triangulation of the
# sphere's surface. `radius`/`resolution` are both in km here (see
# gmshplanet's docstring) - this is a coarse first pass, refined below.
md = pyissm.model.mesh.gmshplanet(md, radius=RADIUS_KM, resolution=150)

# Sanity-check the mesh size before spending time on refinement.
print(f'Initial mesh: {md.mesh.numberofvertices} vertices, {md.mesh.numberofelements} elements')

### Ocean/land classification

`gmtmask` classifies each mesh vertex as ocean or land by testing it against
real Natural Earth coastline polygons. We need this classification twice:
once now, to compute the coastal-refinement metric below, and again after
refining the mesh, since refinement changes which vertices exist.

In [ ]:
from pyissm.data.ocean_mask import gmtmask

# Classify each vertex of the initial (coarse) mesh: 1 = ocean, 0 = land.
ocean = gmtmask(md.mesh.lat, md.mesh.long)

### Coastal-distance refinement metric

`gmshplanet`'s `refine`/`refinemetric` arguments drive local mesh resolution
from a per-vertex target element size, rather than a single global
resolution. We derive that per-vertex target from each vertex's distance to
the coastline, using a `scipy.spatial.cKDTree` nearest-neighbour query on
3-D unit vectors — a single vectorised query over the whole mesh, not the
MATLAB tutorial's O(N²) nested loop over every vertex pair.

`gmtmask` only gives us a per-vertex ocean/land label, not explicit
coastline geometry, so "distance to the coast" is approximated as the
distance from each vertex to the nearest vertex of the *opposite*
classification (nearest land vertex, for an ocean vertex, and vice versa).
At the initial mesh's resolution this is a reasonable proxy for distance to
the true coastline — and that same coastal band is exactly what gets refined
in the next mesh pass anyway, so the approximation only needs to be good
enough to identify it. The resulting distance is then clamped to a
floor/ceiling target element size: finer on the ocean side
(`mindist_coast`) than the land side (`mindist_land`), since the sea-level
solution is driven primarily by ocean loading, and capped at a coarse
`maxdist` far from any coast.

In [ ]:
from scipy.spatial import cKDTree


def _coastal_distance_metric(lat, long, ocean, mindist_coast, mindist_land, maxdist, radius=RADIUS_KM * 1e3):

    """
    Build a per-vertex mesh-refinement target size from distance to the coast.

    Parameters
    ----------
    lat : ndarray
        Vertex latitude, decimal degrees.
    long : ndarray
        Vertex longitude, decimal degrees.
    ocean : ndarray
        Per-vertex ocean/land classification (1 = ocean, 0 = land), e.g. from `gmtmask`.
    mindist_coast : float
        Minimum (finest) target size for ocean vertices, metres.
    mindist_land : float
        Minimum (finest) target size for land vertices, metres.
    maxdist : float
        Maximum (coarsest) target size far from the coast, metres.
    radius : float, optional
        Sphere radius, metres. Defaults to the mean Earth radius used elsewhere in this notebook.

    Returns
    -------
    ndarray
        Per-vertex refinement target size, metres, suitable for `gmshplanet`'s `refinemetric`.
    """

    # Convert every vertex's lat/long to a 3-D unit vector, so "nearest
    # vertex" can be answered with an ordinary Euclidean-distance KD-tree
    # instead of a slower, purpose-built great-circle search.
    lat_r = np.radians(np.ravel(lat))
    long_r = np.radians(np.ravel(long))
    xyz = np.column_stack([np.cos(lat_r) * np.cos(long_r),
                           np.cos(lat_r) * np.sin(long_r),
                           np.sin(lat_r)])

    is_ocean = np.ravel(ocean).astype(bool)

    # For every ocean vertex, find its nearest land vertex, and vice versa -
    # two KD-tree queries (one per class) cover every vertex exactly once.
    chord = np.empty(xyz.shape[0])
    chord[is_ocean], _ = cKDTree(xyz[~is_ocean]).query(xyz[is_ocean])
    chord[~is_ocean], _ = cKDTree(xyz[is_ocean]).query(xyz[~is_ocean])

    # Convert the 3-D straight-line ("chord") distance on the unit sphere
    # back to a great-circle distance in metres.
    dist = 2.0 * np.arcsin(np.clip(chord / 2.0, 0.0, 1.0)) * radius

    # Clamp to a floor/ceiling target element size, with a finer floor on
    # the ocean side than the land side (see the markdown above).
    metric = np.where(is_ocean,
                      np.clip(dist, mindist_coast, maxdist),
                      np.clip(dist, mindist_land, maxdist))
    return metric

### Refined mesh

Rebuild the mesh using the coastal-distance metric above as the target
element size. `gmshplanet`'s refine mode treats the *prior* mesh as an
external input rather than something it can update in place, so `md.mesh`
must first be reset to an empty mesh object, with a separate reference kept
to the prior mesh to pass as `refine=`.

In [ ]:
# Compute the target element size at every vertex of the initial mesh.
dist_metric = _coastal_distance_metric(md.mesh.lat, md.mesh.long, ocean,
                                       mindist_coast=150e3, mindist_land=300e3, maxdist=600e3)

# gmshplanet's refine mode reads the prior mesh from `refine=`, not from
# `md.mesh` - and it rejects a non-empty `md.mesh` outright - so the prior
# mesh must be saved off before md.mesh is reset to empty.
prior_mesh = md.mesh
md.mesh = pyissm.model.classes.mesh.mesh3dsurface()

# Rebuild the mesh, now targeting the variable per-vertex element size
# computed above instead of the initial pass's single uniform resolution.
md = pyissm.model.mesh.gmshplanet(md, radius=RADIUS_KM, resolution=150,
                                  refine=prior_mesh, refinemetric=dist_metric)

# Confirm the refinement had the intended effect: resolution concentrated
# near the coast can mean *fewer* vertices overall, since most of the globe
# is far from any coast and coarsens towards `maxdist`.
print(f'Refined mesh: {md.mesh.numberofvertices} vertices, {md.mesh.numberofelements} elements')

### Ocean level set

Re-classify ocean/land on the refined mesh (refinement changed which
vertices exist, so the earlier classification no longer applies), and store
the result as the model's ocean level set. This is what tells the
solid-earth solver which vertices experience ocean loading. Following
ISSM's level-set sign convention (see `pyissm.model.classes.mask`):
negative inside the ocean, positive on land.

In [ ]:
# Re-classify ocean/land on the refined mesh's vertices.
ocean = gmtmask(md.mesh.lat, md.mesh.long)

# Store as the model's ocean level set, per the ISSM sign convention:
# negative = ocean, positive = land.
md.mask.ocean_levelset = np.where(ocean == 0, 1.0, -1.0)

### Visual check

`pyissm.plot.plot_mesh2d` grids on the model's projected Cartesian `x`/`y`,
which for this global spherical mesh are 3-D coordinates on the sphere
rather than a flat map — it will not produce a meaningful lat/long view
here. A small notebook-local scatter plot on `md.mesh.lat`/`md.mesh.long` is
the correct choice for this global mesh, not a failure to reuse the existing
plotting helpers (the same reasoning applies to the single-epoch and
transient plots in Steps 5 and 7). Look for finer, denser points tracing out
the coastlines, and coarser, sparser points over open ocean and continental
interiors.

In [ ]:
import matplotlib.pyplot as plt

# Colour each vertex by its ocean(1)/land(0) classification; point density
# itself shows the coastal refinement - it should visibly trace coastlines.
fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(md.mesh.long, md.mesh.lat, c=ocean, cmap='coolwarm_r', s=2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Refined mesh vertices ({md.mesh.numberofvertices}), coloured by ocean (1) / land (0)')
fig.colorbar(sc, ax=ax, label='ocean mask')
plt.show()

---
## 2. GRACE forcing: load → mass, geometry

Step 1 built the domain (the mesh) and the ocean/land classification. This
step builds the actual physical *forcing* — the surface mass change that
drives the sea-level response — from a single month of the GRACE gravimetry
product, and assembles the minimal geometry pyISSM needs to run a
mass-transport solve.

GRACE measures mass change as an equivalent thickness of liquid water (how
thick a layer of water would need to be, spread over each grid cell, to
produce the observed gravity change). This tutorial's solidearth solver
expects an *ice*-equivalent thickness change, so the water-equivalent value
from GRACE is converted using the ratio of freshwater to ice density —
the same mass of water occupies a taller column as (less dense) ice.

In [ ]:
from pyissm.data.grace import grace

# A single GRACE epoch, expressed as a decimal year (day 15 of 2007) - this
# tutorial computes the fingerprint for one month before extending to a
# multi-month transient series in Step 6. tmin == tmax selects one epoch.
year_month = 2007 + 15 / 365

# Interpolate that month's water-equivalent load onto every mesh vertex.
# grace() returns metres of water equivalent, with ocean/masked source
# cells already treated as zero load so coastal land queries aren't
# contaminated by adjacent no-data ocean cells (see its docstring).
water_load = grace(md, year_month, year_month, filename=GRACE_NC, onvertex=True)

# Convert water-equivalent thickness to ice-equivalent thickness: the same
# mass of water occupies a taller column as less-dense ice.
rho_w2i = md.materials.rho_freshwater / md.materials.rho_ice
ice_load = water_load * rho_w2i

### Model geometry

This tutorial is about the sea-level response to a mass change, not about
simulating realistic ice-sheet geometry, so the baseline geometry is
deliberately simple and flat: a uniform 100 m ice thickness everywhere, on a
bed at sea level. The GRACE-derived ice load computed above is layered on
top of this flat baseline as a *thickness perturbation* to solve for, not
folded into the baseline itself.

In [ ]:
nv = md.mesh.numberofvertices

# Deliberately flat baseline geometry: bed at sea level, uniform 100 m ice
# thickness everywhere. The GRACE forcing below perturbs this baseline
# rather than describing real ice-sheet topography.
md.geometry.bed = np.zeros(nv)
md.geometry.base = md.geometry.bed.copy()
md.geometry.thickness = 100 * np.ones(nv)
md.geometry.surface = md.geometry.bed + md.geometry.thickness

### Mass-transport forcing

`md.masstransport.spcthickness` is how ISSM specifies a *prescribed*
thickness through time, rather than one computed from ice dynamics. It
needs two time levels here — thickness at the start (unperturbed) and at
the end (perturbed by the GRACE ice load) — plus, per ISSM's constraint
array convention, one extra trailing row holding the time stamp of each
column (`0`, `1`), appended *after* the per-vertex data rather than mixed
into it.

In [ ]:
# Two time levels: the flat baseline, then the same baseline plus the
# GRACE-derived ice load. np.tile duplicates the baseline into both columns
# before the load is added to only the second (final) one.
md.masstransport.spcthickness = np.tile(md.geometry.thickness[:, None], (1, 2))
md.masstransport.spcthickness[:, -1] += ice_load[:, 0]

# ISSM's constraint-array convention: append one trailing row holding the
# time stamp of each column (here, times 0 and 1), after the per-vertex
# thickness data rather than mixed into it.
md.masstransport.spcthickness = np.vstack([md.masstransport.spcthickness, [0, 1]])

# No surface mass balance process is being modelled here - all of the mass
# change comes from the prescribed GRACE forcing above.
md.smb.mass_balance = np.zeros(nv)